# Course 2 — Data Manipulation (Pandas)

Live-coding notebook that mirrors the slide deck.

**Sections**
1. DataFrames & Loading (0:00–0:30)
2. Selection & Filtering (0:30–1:00)
3. Cleaning & Grouping (1:00–1:30)

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
import pandas as pd
import numpy as np
from shared.data_utils import load_dataset
pd.__version__


## 1. DataFrames & Loading

In [ ]:
df = load_dataset('penguins')
df.head()


In [ ]:
df.info()
df.describe(include='all')


### Index vs columns

In [ ]:
print('index:', df.index)
print('cols :', df.columns.tolist())
# Set a more meaningful index
df2 = df.reset_index().rename(columns={'index': 'penguin_id'}).set_index('penguin_id')
df2.head()


## 2. Selection & Filtering

### `.loc` (label) vs `.iloc` (position)

In [ ]:
df.loc[0:3, ['species', 'island', 'bill_length_mm']]


In [ ]:
df.iloc[0:3, 0:3]


### Boolean indexing — the workhorse

In [ ]:
adelie_long = df[(df['species'] == 'Adelie') & (df['bill_length_mm'] > 40)]
adelie_long.head()


In [ ]:
# Cleaner syntax for complex filters
df.query('species == "Adelie" and bill_length_mm > 40').head()


## 3. Cleaning & Grouping

### Missing values

In [ ]:
df.isna().sum()


In [ ]:
clean = df.dropna()
print('before:', len(df), '  after dropna:', len(clean))


Or impute instead of dropping:

In [ ]:
filled = df.copy()
filled['bill_length_mm'] = filled['bill_length_mm'].fillna(filled['bill_length_mm'].median())
filled.isna().sum()


### groupby + agg

In [ ]:
clean.groupby('species')[['bill_length_mm', 'body_mass_g']].mean().round(2)


In [ ]:
clean.groupby(['species', 'sex']).agg(
    n=('bill_length_mm', 'size'),
    mass_mean=('body_mass_g', 'mean'),
    mass_std=('body_mass_g', 'std'),
).round(1)


### Feature engineering — derive a new column

In [ ]:
clean = clean.assign(
    bill_ratio=lambda d: d['bill_length_mm'] / d['bill_depth_mm'],
    mass_kg=lambda d: d['body_mass_g'] / 1000,
)
clean[['species', 'bill_ratio', 'mass_kg']].head()


### Recap
* Treat a DataFrame like a typed, indexed table.
* `.loc` for labels, `.iloc` for positions, boolean masks for filtering.
* Always inspect NaNs before training anything on the data.